# Module 1 — Ingestion: PDF → Neo4j Document Graph

**What we build in this module:**
- Parse 10-K filings with **Docling** (threaded PDF pipeline, table structure, OCR)
- Split each filing into token-aware **chunks** with `HybridChunker`
- Embed each chunk locally with a Hugging Face sentence-transformers model
- Store the result in Neo4j as a `(Company)-[:HAS_DOCUMENT]->(Document)-[:HAS_CHUNK]->(Chunk)` graph, with `Chunk.embedding` populated
- Apply schema constraints and a vector index sized to the embedding model in use
- Wire up a **one-shot RAG baseline**: embed the question, vector-search for chunks, one LLM call to answer

**Services introduced:**
- `services.pdf_parsing_service` — PDF conversion + chunking singleton
- `services.embedding_service` — local Hugging Face embedding singleton
- `services.neo4j_service` — Neo4j driver singleton
- `ingestion.document_loader.load_from_path` — PDF → `list[Document]`
- `ingestion.document_loader.add_embeddings` — `list[Document]` → same list, with `metadata["embedding"]` set
- `ingestion.graph_writer.write_documents` — `list[Document]` → Neo4j
- `ingestion.schema.apply_basic_schema` / `apply_embedding_schema` — constraints + indexes
- `qa.baseline` — one-shot RAG: `embed_question` / `retrieve_chunks` / `build_context` / `generate_answer`

## 1. Setup

In [ ]:
from pathlib import Path

from financial_advisor.ingestion.document_loader import add_embeddings, load_from_path
from financial_advisor.ingestion.graph_writer import write_documents
from financial_advisor.ingestion.schema import apply_basic_schema, apply_embedding_schema
from financial_advisor.services.embedding_service import embedding_service
from financial_advisor.services.pdf_parsing_service import pdf_parsing_service
from financial_advisor.services.neo4j_service import neo4j_service

FILINGS_DIR = Path("../data/filings")

In [ ]:
# Verify Neo4j connectivity before we go further
ok = neo4j_service.check_connection()
print("Neo4j connected:", ok)
assert ok, "Cannot reach Neo4j — check NEO4J_URI / credentials in .env"

## 2. Apply Graph Schema

`apply_basic_schema()` creates uniqueness constraints on `Company.id`, `Document.id`, `Chunk.id`
plus a fulltext index on `Chunk.text`.

`apply_embedding_schema(dimensions)` creates a vector index on `Chunk.embedding`, sized to
whatever embedding model is in use — here, `embedding_service`'s Hugging Face model.

Running these multiple times is safe — all statements use `IF NOT EXISTS`.

A vector index's dimension can't be changed in place. If `chunk_embedding` already exists
from an earlier run with a different embedding model (e.g. 1536-dim Azure OpenAI vectors),
it must be dropped before `apply_embedding_schema` can recreate it at the current model's
dimension. The cell below only drops it when the dimension actually differs.

In [ ]:
existing = neo4j_service.run_query(
    "SHOW INDEXES YIELD name, options WHERE name = 'chunk_embedding' RETURN options"
)
if existing:
    existing_dims = existing[0]["options"]["indexConfig"]["vector.dimensions"]
    if existing_dims != embedding_service.dimensions:
        print(
            f"  chunk_embedding is {existing_dims}-dim, but {embedding_service.model_name} "
            f"produces {embedding_service.dimensions}-dim vectors — dropping to recreate."
        )
        neo4j_service.run_query("DROP INDEX chunk_embedding IF EXISTS")
    else:
        print("  chunk_embedding already matches the current embedding model — nothing to do.")
else:
    print("  No existing chunk_embedding index found.")

In [ ]:
apply_basic_schema()
print(f"Embedding model: {embedding_service.model_name} ({embedding_service.dimensions} dims)")
apply_embedding_schema(embedding_service.dimensions)
print("Schema applied.")

## 3. Discover Filings

Filings follow the naming convention `{COMPANY}_{YEAR}_{TYPE}.pdf`  
e.g. `APPLE_2018_10K.pdf`, `3M_2018_10K.pdf`

In [ ]:
def parse_filing_name(stem: str) -> tuple[str, int] | None:
    """Return (company_id, year) from stems like 'APPLE_2018_10K'."""
    parts = stem.split("_")
    for i, part in enumerate(parts):
        if part.isdigit() and len(part) == 4:
            return "_".join(parts[:i]) or stem, int(part)
    return None


pdfs = sorted(FILINGS_DIR.glob("**/*.pdf"))
filings = []
for pdf in pdfs:
    parsed = parse_filing_name(pdf.stem)
    if parsed:
        company_id, year = parsed
        filings.append({"path": pdf, "company_id": company_id, "year": year})
        print(f"  {pdf.name}  →  company={company_id}, year={year}")
    else:
        print(f"  [SKIP] {pdf.name} — cannot parse company/year")

print(f"\n{len(filings)} filing(s) ready to ingest.")

## 4. Convert a Single Filing with Docling

Before bulk-ingesting everything, let's inspect one filing step by step  
to understand what Docling extracts and how the chunker splits the document.

In [ ]:
# Pick the first filing for the walkthrough
sample = filings[1]
print(f"Processing: {sample['path'].name}")

conv_result = pdf_parsing_service.convert_pdf(sample["path"])
print("Conversion status:", getattr(conv_result, "status", "ok"))

In [ ]:
# Document-level metadata extracted from the PDF
meta = pdf_parsing_service.extract_metadata(conv_result, sample["path"])
for k, v in meta.items():
    print(f"  {k}: {v}")

In [ ]:
# Chunk the document and inspect the first few chunks
chunks = pdf_parsing_service.chunk_document(conv_result)
print(f"Total chunks: {len(chunks)}\n")

for i, chunk in enumerate(chunks[:10]):
    print(f"--- Chunk {i} (pages {chunk.pages}) ---")
    print(chunk.text[:400])
    print()

## 5. Load a Single Filing into Neo4j

`load_from_path` wraps conversion + chunking and returns LangChain `Document` objects  
ready to be passed to `write_documents`.

In [ ]:
documents = load_from_path(
    sample["path"],
    company_id=sample["company_id"],
    year=sample["year"],
)
print(f"Documents (chunks) produced: {len(documents)}")

# Inspect metadata on one document
print("\nSample metadata:")
for k, v in documents[0].metadata.items():
    print(f"  {k}: {v}")

## 5b. Compute Embeddings

`add_embeddings` batches every chunk's text through `embedding_service` in one call and
sets `metadata["embedding"]` on each `Document` in place.

In [ ]:
documents = add_embeddings(documents)

sample_embedding = documents[0].metadata["embedding"]
print(f"Vector length: {len(sample_embedding)}")
print(f"First 5 values: {sample_embedding[:5]}")

In [ ]:
# Write to Neo4j
write_documents(documents, company_id=sample["company_id"])
print("Done.")

## 6. Verify — Query the Graph

Check that the nodes and relationships were created correctly.

In [ ]:
# Node counts
counts = neo4j_service.run_query("""
    MATCH (c:Company) WITH count(c) AS companies
    MATCH (d:Document) WITH companies, count(d) AS documents
    MATCH (ch:Chunk)   RETURN companies, documents, count(ch) AS chunks
""")
print(counts[0])

In [ ]:
# Sample the graph: one company, its documents, first 3 chunks
rows = neo4j_service.run_query("""
    MATCH (c:Company)-[:HAS_DOCUMENT]->(d:Document)-[:HAS_CHUNK]->(ch:Chunk)
    WHERE c.id = $company_id
    RETURN c.id AS company, d.doc_name AS document, d.total_pages AS pages,
           ch.idx AS chunk_idx, ch.pages AS chunk_pages,
           size(ch.embedding) AS embedding_size,
           left(ch.text, 120) AS snippet
    ORDER BY ch.idx
    LIMIT 3
""", {"company_id": sample["company_id"]})

for row in rows:
    print(row)

## 7. Ingest All Filings

Now run the full loop over every PDF found in `data/filings/`.  
Already-ingested documents are skipped automatically (idempotent).

In [ ]:
for filing in filings:
    print(f"\n[{filing['company_id']}] {filing['path'].name} (year={filing['year']})")
    docs = load_from_path(
        filing["path"],
        company_id=filing["company_id"],
        year=filing["year"],
    )
    if not docs:
        print("  No chunks produced — skipping.")
        continue
    print(f"  {len(docs)} chunks extracted")
    docs = add_embeddings(docs)
    write_documents(docs, company_id=filing["company_id"])

print("\nAll filings processed.")

In [ ]:
# Final count across all companies
summary = neo4j_service.run_query("""
    MATCH (c:Company)-[:HAS_DOCUMENT]->(d:Document)
    OPTIONAL MATCH (d)-[:HAS_CHUNK]->(ch:Chunk)
    RETURN c.id AS company, d.doc_name AS document,
           d.total_pages AS pages, count(ch) AS chunks
    ORDER BY company, document
""")

for row in summary:
    print(row)

## 8. One-Shot RAG Baseline — Ask a Question

The simplest retrieval-augmented pattern: embed the question with the same model used
for the chunks, pull back the top-k most similar chunks from the `chunk_embedding`
vector index, and hand them to the LLM in a single call. No tool use, no iteration —
one retrieval, one generation.

Implementation lives in `financial_advisor.qa.baseline`, split into four steps so each
can be inspected on its own: `embed_question`, `retrieve_chunks`, `build_context`,
`generate_answer`.

In [ ]:
from financial_advisor.qa.baseline import (
    build_context,
    embed_question,
    generate_answer,
    retrieve_chunks,
)

### 8a. Step by Step, on One Question

In [ ]:
question = "What is 3M's principal executive office address?"
print(f"Question: {question}")

In [ ]:
# Step 1 — embed the question with the same model used for chunks
query_vector = embed_question(question)
print(f"Vector length: {len(query_vector)}")
print(f"First 5 values: {query_vector[:5]}")

In [ ]:
# Step 2 — vector search: top-k chunks by cosine similarity
retrieved = retrieve_chunks(query_vector, k=5)
print(f"Retrieved {len(retrieved)} chunks:\n")
for chunk in retrieved:
    print("#####################")
    print(f"  Score: {chunk.score:.3f}, Doc: {chunk.doc_id}, Chunk ID: {chunk.id}")
    print(f"    {chunk.text[:500]}")
    print("#####################\n")

In [ ]:
# Step 3 — stitch the retrieved chunks into a single context block
context = build_context(retrieved)
print(f"Context: {len(context)} chars from {len(retrieved)} chunks\n")
print(context[:500])

In [ ]:
# Step 4 — one LLM call over the retrieved context: the "one shot" in one-shot RAG
answer = generate_answer(question, retrieved)
print(f"Answer:\n{answer}")

### 8b. A Few More Questions — Easy vs. Harder

Single-chunk factual questions tend to work well: the right chunk scores highest and the
LLM just reads it off. Questions that need reasoning *across* documents are where the
one-shot pattern starts to visibly struggle — there's no mechanism here to run a second,
more targeted retrieval or to reconcile conflicting/partial results. That gap is exactly
what Module 2's agentic loop is built to close.

In [ ]:
EXAMPLE_QUESTIONS = [
    "What is 3M's principal executive office address?",  # single-chunk factual — easy
    "What was Apple's total revenue in fiscal year 2018?",  # single-chunk factual — easy
    "Compare 3M's and Apple's approach to research and development investment, "
    "based on their 2018 10-Ks.",  # cross-document synthesis — hard
]

for q in EXAMPLE_QUESTIONS:
    print(f"\n{'=' * 80}")
    print(f"Q: {q}\n")

    chunks = retrieve_chunks(embed_question(q), k=5)
    print("Retrieved:")
    for chunk in chunks:
        print(f"  [{chunk.score:.3f}] {chunk.doc_id}")

    answer = generate_answer(q, chunks)
    print(f"\nA: {answer}")